# =========================================================
# IMPORTS
# =========================================================

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================================================
# LOAD DATA PREPROCESSED (.npy)
# =========================================================

In [2]:
X_train = np.load("../data/X_train.npy")
X_val   = np.load("../data/X_val.npy")
X_test  = np.load("../data/X_test.npy")

y_train = np.load("../data/y_train.npy")
y_val   = np.load("../data/y_val.npy")
y_test  = np.load("../data/y_test.npy")

print("Train :", X_train.shape)
print("Val   :", X_val.shape)
print("Test  :", X_test.shape)

Train : (237280, 27)
Val   : (50846, 27)
Test  : (50846, 27)


# =========================================================
# MODEL ARCHITECTURE (MLP)
# =========================================================

In [3]:
model = Sequential([

    # Input layer + first hidden layer
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),

    # Second hidden layer
    Dense(64, activation="relu"),
    Dropout(0.3),

    # Third hidden layer (améliore la capacité)
    Dense(32, activation="relu"),
    Dropout(0.2),

    # Output layer (régression)
    Dense(1)

])

c:\Users\ALPHA\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# =========================================================
# COMPILATION
# =========================================================

In [4]:
model.compile(

    optimizer="adam",

    loss="mse",

    metrics=["mae"]

)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,953 (54.50 KB)

 Trainable params: 13,953 (54.50 KB)

 Non-trainable params: 0 (0.00 B)

# =========================================================
# CALLBACKS (STABILITY + PERFORMANCE)
# =========================================================

In [5]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True

)

lr_scheduler = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=3,

    verbose=1

)

# =========================================================
# TRAINING MODEL
# =========================================================


In [6]:
history = model.fit(

    X_train,
    y_train,

    validation_data=(X_val, y_val),

    epochs=50,

    batch_size=32,

    callbacks=[early_stop, lr_scheduler],

    verbose=1

)
import pickle

with open("../models/history.pkl", "wb") as f:
    pickle.dump(history.history, f)

Epoch 1/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - loss: 76.8612 - mae: 5.8702 - val_loss: 55.0462 - val_mae: 4.9619 - learning_rate: 0.0010
Epoch 2/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 61.6867 - mae: 5.3270 - val_loss: 52.6558 - val_mae: 4.9651 - learning_rate: 0.0010
Epoch 3/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 59.6827 - mae: 5.2406 - val_loss: 59.4776 - val_mae: 5.0808 - learning_rate: 0.0010
Epoch 4/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - loss: 58.2351 - mae: 5.1767 - val_loss: 51.5319 - val_mae: 4.8588 - learning_rate: 0.0010
Epoch 5/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - loss: 57.3059 - mae: 5.1331 - val_loss: 51.9291 - val_mae: 4.9396 - learning_rate: 0.0010
Epoch 6/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 56.6713 - mae: 5.0966 - val_loss: 51.2510 - val_mae: 4.8645 - learning_rate: 0.0010
Epoch 7/50
7415/7415 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 55.9625 - mae: 5.0674 - val_loss: 51.7943 - val_mae: 4.

# =========================================================
# SAVE TRAINED MODEL
# =========================================================


In [ ]:
model.save("../models/mlp_model.keras")

print("Model saved successfully ✔")

Model saved successfully ✔
